# Mini Project I-Smile: EDA to Machine Learning
**Kelompok 7**

**Judul:** Prediksi Risiko Depresi Remaja Berdasarkan Penggunaan Media Sosial dan Gaya Hidup

**Jenis Masalah:** Klasifikasi (Binary Classification)

**Target:** `depression_label` (0 = Tidak Depresi, 1 = Depresi)

**Dataset:** [Teen Mental Health Dataset — Kaggle](https://www.kaggle.com/datasets/sunil123kumar/social-media-impact-on-mental-health?select=Teen_Mental_Health_Dataset.csv)

**Pertanyaan yang ingin dijawab:** Berdasarkan pola penggunaan media sosial dan gaya hidup remaja (jam penggunaan medsos, platform, jam tidur, screen time, performa akademik, aktivitas fisik, tingkat stres, kecemasan, dan kecanduan), dapatkah kita memprediksi apakah seorang remaja berisiko mengalami depresi?

**Siapa yang memakai hasilnya:** Konselor sekolah, psikolog, dan orang tua yang ingin melakukan deteksi dini risiko depresi pada remaja.

## 1. Import Library & Load Data

In [ ]:
# Import library yang dibutuhkan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import (train_test_split, cross_val_score,
                                      GridSearchCV, StratifiedKFold)
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              confusion_matrix, ConfusionMatrixDisplay,
                              classification_report)

# Boosting
from xgboost import XGBClassifier

print("Semua library berhasil diimport!")

In [ ]:
# Load dataset
# Sumber: https://www.kaggle.com/datasets/sunil123kumar/social-media-impact-on-mental-health
import os

if os.path.exists('Teen_Mental_Health_Dataset.csv'):
    path_data = 'Teen_Mental_Health_Dataset.csv'
elif os.path.exists('/content/Teen_Mental_Health_Dataset.csv'):
    path_data = '/content/Teen_Mental_Health_Dataset.csv'
else:
    path_data = 'Teen_Mental_Health_Dataset.csv'

df = pd.read_csv(path_data)
print(f"Dataset berhasil dimuat: {df.shape[0]} baris, {df.shape[1]} kolom")
display(df.head())

## 2. Exploratory Data Analysis (EDA)

### 2.1 Ringkasan Data
Langkah pertama adalah memahami struktur, tipe data, dan kondisi dataset secara keseluruhan.

In [ ]:
# Informasi dataset: tipe data dan jumlah non-null
print("=" * 60)
print("INFORMASI DATASET")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("STATISTIK DESKRIPTIF")
print("=" * 60)
display(df.describe())

print("\n" + "=" * 60)
print("PENGECEKAN MISSING VALUE")
print("=" * 60)
print(df.isnull().sum())

print("\n" + "=" * 60)
print("PENGECEKAN DUPLIKAT")
print("=" * 60)
print(f"Jumlah baris duplikat: {df.duplicated().sum()}")

print("\n" + "=" * 60)
print("DISTRIBUSI KOLOM KATEGORIK")
print("=" * 60)
for col in ['gender', 'platform_usage', 'social_interaction_level']:
    print(f"\n{col}:")
    print(df[col].value_counts())

print("\n" + "=" * 60)
print("DISTRIBUSI TARGET (depression_label)")
print("=" * 60)
print(df['depression_label'].value_counts())
print(f"\nProporsi kelas 1 (Depresi): {df['depression_label'].mean():.2%}")

**Temuan Ringkasan Data:**
1. Dataset memiliki **1200 baris** dan **13 kolom** (10 numerik, 3 kategorik).
2. **Tidak ada missing value** di seluruh kolom — dataset sudah bersih.
3. Perlu dicek apakah ada **duplikat** yang bisa memengaruhi model.
4. Kolom kategorik: `gender` (male/female), `platform_usage` (Instagram/TikTok/Both), `social_interaction_level` (low/medium/high).
5. Target `depression_label` bersifat **binary** (0 dan 1).

### 2.2 Visualisasi

Minimal 3 visualisasi dengan temuan masing-masing untuk memahami pola dan distribusi data.

#### Visualisasi 1: Distribusi Target (`depression_label`)

In [ ]:
# Visualisasi 1: Distribusi target depression_label
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='depression_label', hue='depression_label',
                   palette='Set2', legend=False)
plt.title('Distribusi Target: Depression Label', fontsize=14, fontweight='bold')
plt.xlabel('Depression Label (0 = Tidak Depresi, 1 = Depresi)')
plt.ylabel('Jumlah')

# Tambah label angka di atas bar
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Hitung proporsi
n_depresi = df['depression_label'].sum()
n_total = len(df)
print(f"Kelas 0 (Tidak Depresi): {n_total - n_depresi} ({(n_total - n_depresi)/n_total:.1%})")
print(f"Kelas 1 (Depresi)      : {n_depresi} ({n_depresi/n_total:.1%})")

**Temuan Visualisasi 1:**
- Dataset sangat **tidak seimbang (imbalanced)**. Kelas 0 (Tidak Depresi) mendominasi dengan sekitar **97.4%**, sedangkan kelas 1 (Depresi) hanya sekitar **2.6%** (±31 dari 1200 data).
- Implikasi: Model yang selalu menebak "Tidak Depresi" sudah punya accuracy ~97.4%. Oleh karena itu, **accuracy saja tidak cukup** untuk menilai performa model. Kita perlu melihat **precision** dan **recall** juga.
- Saat split data, kita wajib menggunakan `stratify=y` agar proporsi kelas tetap sama di data latih dan data uji.

#### Visualisasi 2: Heatmap Korelasi Fitur Numerik

In [ ]:
# Visualisasi 2: Heatmap korelasi fitur numerik
plt.figure(figsize=(12, 9))
numeric_df = df.select_dtypes(include=['int64', 'float64'])
corr_matrix = numeric_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Mask segitiga atas
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f',
            mask=mask, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Heatmap Korelasi Fitur Numerik', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Temuan Visualisasi 2:**
- `stress_level`, `anxiety_level`, dan `addiction_level` **tidak berkorelasi kuat** dengan `depression_label` secara linear. Ini menarik karena secara logis kita mengira mereka berkaitan erat.
- Korelasi antar fitur umumnya rendah, yang berarti fitur-fitur cukup independen satu sama lain — bagus untuk pemodelan.
- Perlu diperhatikan bahwa korelasi linear tidak menangkap hubungan non-linear. Model berbasis pohon (Decision Tree, Random Forest, dll.) bisa menangkap pola non-linear ini.

#### Visualisasi 3: Distribusi Fitur Numerik (Histogram)

In [ ]:
# Visualisasi 3: Histogram distribusi fitur numerik
fitur_numerik = ['age', 'daily_social_media_hours', 'sleep_hours',
                 'screen_time_before_sleep', 'academic_performance',
                 'physical_activity', 'stress_level', 'anxiety_level', 'addiction_level']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Distribusi Fitur Numerik', fontsize=16, fontweight='bold', y=1.02)

for i, col in enumerate(fitur_numerik):
    ax = axes[i // 3, i % 3]
    ax.hist(df[col], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    ax.set_title(col, fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('Frekuensi')

plt.tight_layout()
plt.show()

**Temuan Visualisasi 3:**
- `age`: Distribusi cukup merata dari 13–19 tahun.
- `daily_social_media_hours`: Distribusi relatif merata, tidak terlalu skew.
- `sleep_hours`: Distribusi mendekati normal, rata-rata sekitar 6.4 jam.
- `stress_level`, `anxiety_level`, `addiction_level`: Distribusi cukup merata (uniform-like) dari 1–10.
- `physical_activity`: Distribusi merata dari 0–2 jam.
- Sebagian besar fitur tidak menunjukkan skewness ekstrem, sehingga tidak perlu transformasi khusus.

#### Visualisasi 4: Deteksi Outlier (Boxplot)

In [ ]:
# Visualisasi 4: Boxplot untuk deteksi outlier
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Boxplot Fitur Numerik (Deteksi Outlier)', fontsize=16, fontweight='bold', y=1.02)

for i, col in enumerate(fitur_numerik):
    ax = axes[i // 3, i % 3]
    ax.boxplot(df[col], vert=True, patch_artist=True,
               boxprops=dict(facecolor='lightblue'))
    ax.set_title(col, fontsize=11)

plt.tight_layout()
plt.show()

# Hitung outlier per kolom dengan metode IQR
print("Jumlah outlier per fitur (metode IQR):")
print("-" * 45)
for col in fitur_numerik:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outlier = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"  {col}: {n_outlier} outlier")

**Temuan Visualisasi 4:**
- Berdasarkan metode IQR, sebagian besar fitur **tidak memiliki outlier signifikan** atau hanya sedikit.
- Ini masuk akal karena data sudah dikurasi dengan baik (rentang nilai sudah terbatas, misalnya `stress_level` 1–10, `sleep_hours` 4–9).
- **Keputusan: Tidak menghapus outlier**, karena nilai-nilai tersebut masih dalam rentang wajar untuk data kesehatan mental remaja.

#### Visualisasi 5: Platform Penggunaan Media Sosial vs Depresi

In [ ]:
# Visualisasi 5: Platform usage vs depression_label
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Kiri: Countplot per platform
sns.countplot(data=df, x='platform_usage', hue='depression_label',
              palette='Set2', ax=axes[0])
axes[0].set_title('Jumlah per Platform & Label Depresi', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Platform')
axes[0].set_ylabel('Jumlah')
axes[0].legend(title='Depression', labels=['Tidak Depresi', 'Depresi'])

# Kanan: Persentase depresi per platform
depresi_rate = df.groupby('platform_usage')['depression_label'].mean() * 100
depresi_rate.plot(kind='bar', color=['#66c2a5', '#fc8d62', '#8da0cb'], ax=axes[1],
                  edgecolor='black')
axes[1].set_title('Persentase Depresi per Platform', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Platform')
axes[1].set_ylabel('Persentase Depresi (%)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

# Tambah label persen
for i, v in enumerate(depresi_rate):
    axes[1].text(i, v + 0.1, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

**Temuan Visualisasi 5:**
- Distribusi pengguna antar platform (Instagram, TikTok, Both) cukup merata.
- Persentase remaja yang depresi tampak tidak berbeda jauh antar platform.
- Artinya, **jenis platform saja bukan faktor utama** penentu depresi pada dataset ini.

#### Visualisasi 6: Gender vs Depresi

In [ ]:
# Visualisasi 6: Gender vs depression_label
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Kiri: Countplot
sns.countplot(data=df, x='gender', hue='depression_label',
              palette='Set2', ax=axes[0])
axes[0].set_title('Jumlah per Gender & Label Depresi', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Jumlah')
axes[0].legend(title='Depression', labels=['Tidak Depresi', 'Depresi'])

# Kanan: Persentase depresi per gender
depresi_gender = df.groupby('gender')['depression_label'].mean() * 100
depresi_gender.plot(kind='bar', color=['#8da0cb', '#fc8d62'], ax=axes[1],
                    edgecolor='black')
axes[1].set_title('Persentase Depresi per Gender', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Persentase Depresi (%)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

for i, v in enumerate(depresi_gender):
    axes[1].text(i, v + 0.1, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

**Temuan Visualisasi 6:**
- Jumlah remaja laki-laki dan perempuan dalam dataset cukup seimbang.
- Persentase depresi antar gender juga **tidak berbeda drastis**, meskipun mungkin ada sedikit perbedaan.
- Gender tetap dimasukkan sebagai fitur karena bisa memberikan kontribusi kecil pada model.

## 3. Data Cleaning

Setiap langkah cleaning disertai **alasan (reasoning)** agar prosesnya transparan dan dapat dipertanggungjawabkan.

### 3.1 Penanganan Missing Value

In [ ]:
# Cek missing value
print("Jumlah Missing Value per Kolom:")
print("-" * 40)
print(df.isnull().sum())
print(f"\nTotal missing value: {df.isnull().sum().sum()}")

**Keputusan:** Tidak ada tindakan yang dilakukan.

**Alasan:** Tidak ada missing value di seluruh kolom (total = 0). Dataset sudah lengkap sehingga tidak perlu dilakukan imputasi (pengisian nilai) atau penghapusan baris.

### 3.2 Penanganan Duplikat

In [ ]:
# Cek dan tangani duplikat
n_duplikat = df.duplicated().sum()
print(f"Jumlah baris duplikat: {n_duplikat}")

if n_duplikat > 0:
    print(f"\nContoh baris duplikat:")
    display(df[df.duplicated(keep=False)].head(10))
    print(f"\nMenghapus {n_duplikat} baris duplikat...")
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Ukuran dataset setelah hapus duplikat: {df.shape}")
else:
    print("Tidak ada baris duplikat. Dataset tetap utuh.")

**Keputusan:** Cek duplikat — jika ditemukan, hapus.

**Alasan:** Baris duplikat bisa membuat model "menghafal" data tertentu (overfitting) dan memberikan bobot lebih pada observasi yang sama. Menghapus duplikat memastikan setiap observasi unik dan representatif.

### 3.3 Penanganan Outlier

In [ ]:
# Analisis outlier dengan metode IQR
print("Analisis Outlier (Metode IQR):")
print("=" * 55)
for col in fitur_numerik:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col:30s} | IQR: {IQR:.2f} | Batas: [{lower:.2f}, {upper:.2f}] | Outlier: {len(outliers)}")

**Keputusan:** Tidak menghapus outlier.

**Alasan:**
1. Berdasarkan analisis IQR, jumlah outlier sangat sedikit atau bahkan tidak ada di sebagian besar fitur.
2. Data ini berkaitan dengan kesehatan mental remaja, di mana nilai-nilai "ekstrem" (misalnya tingkat stres atau kecemasan yang sangat tinggi) bisa jadi merupakan **informasi penting, bukan noise**.
3. Model berbasis pohon (Decision Tree, Random Forest, XGBoost) secara alami **tahan terhadap outlier** karena membuat keputusan berdasarkan pemisahan biner, bukan perhitungan jarak.
4. Untuk KNN yang sensitif terhadap skala, kita akan menggunakan **StandardScaler** yang mengurangi dampak outlier.

### 3.4 Encoding Kolom Kategorik

In [ ]:
# Buat copy untuk preprocessing
df_clean = df.copy()

# 1. Encoding 'gender' (binary: male/female -> 0/1)
# Gender bersifat nominal tapi hanya 2 nilai, jadi cukup di-map ke 0/1
df_clean['gender'] = df_clean['gender'].map({'male': 0, 'female': 1})
print("Encoding gender: male=0, female=1")

# 2. Encoding 'social_interaction_level' (ordinal: low < medium < high)
# Kolom ini bersifat ordinal (ada urutan), jadi kita map secara manual
df_clean['social_interaction_level'] = df_clean['social_interaction_level'].map({
    'low': 0, 'medium': 1, 'high': 2
})
print("Encoding social_interaction_level: low=0, medium=1, high=2")

# 3. Encoding 'platform_usage' (nominal: Instagram/TikTok/Both)
# Kolom ini bersifat nominal (tidak ada urutan), jadi kita pakai One-Hot Encoding
df_clean = pd.get_dummies(df_clean, columns=['platform_usage'], drop_first=False)
# Konversi kolom boolean hasil get_dummies ke int (0/1)
for col in df_clean.columns:
    if df_clean[col].dtype == 'bool':
        df_clean[col] = df_clean[col].astype(int)
print("Encoding platform_usage: One-Hot Encoding (3 kolom baru)")

print(f"\nUkuran dataset setelah encoding: {df_clean.shape}")
print(f"Kolom: {list(df_clean.columns)}")
display(df_clean.head())

**Alasan Encoding:**
1. **`gender`** → Binary mapping (0/1): Karena hanya ada 2 nilai (male/female), cukup diubah ke angka 0 dan 1. Hasilnya sama dengan One-Hot Encoding tapi lebih ringkas.
2. **`social_interaction_level`** → Ordinal mapping (0/1/2): Kolom ini memiliki **urutan alami** (low < medium < high), sehingga kita map secara manual agar model menangkap urutan tersebut. Menggunakan LabelEncoder atau One-Hot di sini akan kehilangan informasi urutan.
3. **`platform_usage`** → One-Hot Encoding (`pd.get_dummies`): Kolom ini bersifat **nominal** (Instagram, TikTok, Both tidak punya urutan), sehingga One-Hot Encoding lebih tepat daripada LabelEncoder yang akan memberi urutan palsu (misalnya Both=0, Instagram=1, TikTok=2 seolah-olah TikTok > Instagram).

## 4. Split Data

Data dibagi menjadi **data latih (80%)** dan **data uji (20%)** dengan `random_state=42` untuk reproduksibilitas.

In [ ]:
# Pisahkan fitur (X) dan target (y)
X = df_clean.drop('depression_label', axis=1)
y = df_clean['depression_label']

# Split data: 80% latih, 20% uji
# stratify=y WAJIB untuk klasifikasi agar proporsi kelas sama di train dan test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y    # PENTING: menjaga proporsi kelas di train dan test
)

print(f"Jumlah data latih : {X_train.shape[0]} ({X_train.shape[0]/len(X):.0%})")
print(f"Jumlah data uji   : {X_test.shape[0]} ({X_test.shape[0]/len(X):.0%})")
print(f"Jumlah fitur       : {X_train.shape[1]}")
print(f"\nProporsi kelas di data latih:")
print(f"  Kelas 0 (Tidak Depresi): {(y_train == 0).sum()} ({(y_train == 0).mean():.1%})")
print(f"  Kelas 1 (Depresi)      : {(y_train == 1).sum()} ({(y_train == 1).mean():.1%})")
print(f"\nProporsi kelas di data uji:")
print(f"  Kelas 0 (Tidak Depresi): {(y_test == 0).sum()} ({(y_test == 0).mean():.1%})")
print(f"  Kelas 1 (Depresi)      : {(y_test == 1).sum()} ({(y_test == 1).mean():.1%})")

**Penjelasan:**
- `test_size=0.2`: 20% data untuk uji, 80% untuk latih (sesuai brief).
- `random_state=42`: Agar hasil bisa direproduksi oleh siapa saja.
- `stratify=y`: **Wajib untuk klasifikasi**, terutama karena data kita sangat imbalanced. Tanpa stratify, bisa saja semua data kelas 1 (depresi) masuk ke satu bagian saja, sehingga model tidak bisa belajar/diuji dengan baik.

## 5. Model Standar: KNN + Scaling (Wajib)

Sesuai brief, model standar wajib untuk klasifikasi adalah **KNeighborsClassifier dengan scaling**. KNN membutuhkan scaling karena ia menghitung **jarak antar data**. Tanpa scaling, kolom dengan skala besar (misalnya `stress_level` 1–10) akan mendominasi kolom berskala kecil (misalnya `physical_activity` 0–2).

Kita menggunakan `StandardScaler` yang mengubah setiap kolom agar rata-ratanya 0 dan standar deviasinya 1.

In [ ]:
# Scaling menggunakan StandardScaler
# PENTING: fit hanya di data latih, lalu transform data latih DAN data uji
# Ini mencegah "kebocoran" informasi data uji ke proses pelatihan
scaler = StandardScaler()
scaler.fit(X_train)                            # Cari mean dan std dari data latih saja
X_train_scaled = scaler.transform(X_train)     # Ubah data latih
X_test_scaled = scaler.transform(X_test)       # Ubah data uji dengan mean/std yang SAMA

print("Scaling selesai!")
print(f"Contoh mean fitur sebelum scaling (data latih): {X_train.iloc[:, 0].mean():.4f}")
print(f"Contoh mean fitur setelah scaling (data latih): {X_train_scaled[:, 0].mean():.4f}")
print(f"Contoh std fitur setelah scaling (data latih) : {X_train_scaled[:, 0].std():.4f}")

In [ ]:
# Model Standar: KNN dengan k=5 (default) + StandardScaler
knn_standar = KNeighborsClassifier(n_neighbors=5)
knn_standar.fit(X_train_scaled, y_train)

# Prediksi di data latih DAN data uji
y_train_pred_knn = knn_standar.predict(X_train_scaled)
y_test_pred_knn = knn_standar.predict(X_test_scaled)

# Evaluasi: bandingkan skor latih vs uji
print("=" * 60)
print("EVALUASI MODEL STANDAR: KNN (k=5) + StandardScaler")
print("=" * 60)
print(f"{'Metrik':<15} {'Data Latih':>12} {'Data Uji':>12}")
print("-" * 60)
print(f"{'Accuracy':<15} {accuracy_score(y_train, y_train_pred_knn):>12.4f} {accuracy_score(y_test, y_test_pred_knn):>12.4f}")
print(f"{'Precision':<15} {precision_score(y_train, y_train_pred_knn, zero_division=0):>12.4f} {precision_score(y_test, y_test_pred_knn, zero_division=0):>12.4f}")
print(f"{'Recall':<15} {recall_score(y_train, y_train_pred_knn, zero_division=0):>12.4f} {recall_score(y_test, y_test_pred_knn, zero_division=0):>12.4f}")

print("\n" + "=" * 60)
print("Classification Report (Data Uji):")
print("=" * 60)
print(classification_report(y_test, y_test_pred_knn, zero_division=0))

In [ ]:
# Confusion Matrix KNN Standar
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion Matrix Data Latih
cm_train = confusion_matrix(y_train, y_train_pred_knn)
ConfusionMatrixDisplay(cm_train, display_labels=['Tidak Depresi', 'Depresi']).plot(
    cmap='Blues', colorbar=False, ax=axes[0])
axes[0].set_title('KNN (k=5) — Data Latih', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Prediksi')
axes[0].set_ylabel('Aktual')

# Confusion Matrix Data Uji
cm_test = confusion_matrix(y_test, y_test_pred_knn)
ConfusionMatrixDisplay(cm_test, display_labels=['Tidak Depresi', 'Depresi']).plot(
    cmap='Oranges', colorbar=False, ax=axes[1])
axes[1].set_title('KNN (k=5) — Data Uji', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Prediksi')
axes[1].set_ylabel('Aktual')

plt.tight_layout()
plt.show()

## 6. Tuning Hyperparameter KNN (GridSearchCV)

Kita belum tahu apakah k=5 adalah pilihan terbaik. `GridSearchCV` akan mencoba berbagai nilai k dan memilih yang memberikan skor cross-validation terbaik.

Nilai k yang dicoba **ganjil semua** agar voting tidak mungkin seri pada masalah 2 kelas.

In [ ]:
# GridSearchCV untuk mencari k terbaik
param_grid = {'n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21]}

grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy',
    return_train_score=True
)
grid_knn.fit(X_train_scaled, y_train)

# Hasil
print(f"K terbaik: {grid_knn.best_params_['n_neighbors']}")
print(f"Skor CV terbaik: {grid_knn.best_score_:.4f}")

# Tampilkan semua hasil
hasil_grid = pd.DataFrame(grid_knn.cv_results_)
print("\nHasil GridSearchCV:")
print("-" * 50)
for _, row in hasil_grid.iterrows():
    k_val = row['param_n_neighbors']
    mean = row['mean_test_score']
    std = row['std_test_score']
    print(f"  k={k_val:2d} -> Accuracy CV: {mean:.4f} +/- {std:.4f}")

k_terbaik = grid_knn.best_params_['n_neighbors']

In [ ]:
# Plot accuracy vs k
plt.figure(figsize=(10, 5))
k_values = param_grid['n_neighbors']
mean_scores = hasil_grid['mean_test_score']
std_scores = hasil_grid['std_test_score']

plt.plot(k_values, mean_scores, 'bo-', linewidth=2, markersize=8)
plt.fill_between(k_values, mean_scores - std_scores, mean_scores + std_scores,
                 alpha=0.2, color='blue')
plt.axvline(x=k_terbaik, color='red', linestyle='--', label=f'k terbaik = {k_terbaik}')
plt.xlabel('Nilai k (jumlah tetangga)', fontsize=12)
plt.ylabel('Accuracy (Cross-Validation)', fontsize=12)
plt.title('Pengaruh Nilai k terhadap Accuracy KNN', fontsize=14, fontweight='bold')
plt.xticks(k_values)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Temuan Tuning KNN:**
- GridSearchCV menguji 11 nilai k (1 sampai 21, ganjil) dengan 5-fold stratified cross-validation.
- Nilai k terbaik ditentukan berdasarkan skor accuracy rata-rata tertinggi di cross-validation.
- Shaded area pada grafik menunjukkan ± 1 standar deviasi, yang menggambarkan **variabilitas** skor antar lipatan.
- k=1 cenderung **overfitting** (terlalu sensitif terhadap noise), sementara k yang terlalu besar bisa **underfitting** (kehilangan pola lokal).

## 7. Model Lain & Cross-Validation 5 Lipatan

Semua model dinilai dengan **aturan yang sama**: data latih yang sama, 5-fold stratified cross-validation, dan pengaturan bawaan. KNN dibungkus dalam `Pipeline` bersama `StandardScaler` agar scaling otomatis dan tidak bocor ke data validasi.

Alasan pemilihan 4 model ini:

| Model | Jenis | Butuh Scaling? | Alasan Dipilih |
|---|---|---|---|
| **KNN (k terbaik)** | Distance-based | ✅ Ya | Model standar wajib sesuai brief |
| **Decision Tree** | Tree-based | ❌ Tidak | White box, interpretable, bisa divisualisasi pohonnya |
| **Random Forest** | Ensemble (bagging) | ❌ Tidak | Paling andal untuk data tabular kecil-menengah |
| **XGBoost** | Ensemble (boosting) | ❌ Tidak | Perbandingan: boosting vs bagging (Random Forest) |

**Catatan:** LightGBM tidak dipakai karena redundan dengan XGBoost pada dataset kecil (1200 baris). MLP (Neural Network) juga tidak dipakai karena butuh data jauh lebih banyak untuk unggul.

In [ ]:
# Definisikan semua model
# KNN dibungkus dalam Pipeline agar scaling otomatis
models = {
    f'KNN (k={k_terbaik})': make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=k_terbaik)
    ),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'),
}

# Cross-validation 5 lipatan (Stratified karena data imbalanced)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("=" * 65)
print("CROSS-VALIDATION 5 LIPATAN (DATA LATIH)")
print("=" * 65)
print(f"{'Model':<20} {'CV Mean':>10} {'CV Std':>10}")
print("-" * 65)

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name:<20} {scores.mean():>10.4f} {scores.std():>10.4f}")

In [ ]:
# Latih setiap model di SELURUH data latih, lalu evaluasi di data latih dan data uji
print("=" * 85)
print("EVALUASI TRAIN vs TEST (SEMUA MODEL)")
print("=" * 85)
print(f"{'Model':<20} {'Acc Train':>10} {'Acc Test':>10} {'Prec Test':>10} {'Rec Test':>10}")
print("-" * 85)

eval_results = {}
trained_models = {}

for name, model in models.items():
    # Latih di seluruh data latih
    model.fit(X_train, y_train)
    trained_models[name] = model

    # Prediksi
    y_tr_pred = model.predict(X_train)
    y_te_pred = model.predict(X_test)

    # Hitung metrik
    acc_tr = accuracy_score(y_train, y_tr_pred)
    acc_te = accuracy_score(y_test, y_te_pred)
    prec_te = precision_score(y_test, y_te_pred, zero_division=0)
    rec_te = recall_score(y_test, y_te_pred, zero_division=0)

    eval_results[name] = {
        'Accuracy Train': acc_tr,
        'Accuracy Test': acc_te,
        'Precision Test': prec_te,
        'Recall Test': rec_te,
        'CV Mean': cv_results[name].mean(),
        'CV Std': cv_results[name].std(),
    }

    print(f"{name:<20} {acc_tr:>10.4f} {acc_te:>10.4f} {prec_te:>10.4f} {rec_te:>10.4f}")

## 8. Perbandingan Semua Model

In [ ]:
# Tabel perbandingan lengkap
df_eval = pd.DataFrame(eval_results).T
df_eval.index.name = 'Model'
print("Tabel Perbandingan Lengkap:")
display(df_eval.style.format('{:.4f}').highlight_max(axis=0, color='lightgreen'))

In [ ]:
# Barplot perbandingan skor CV dan Train vs Test
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

model_names = list(cv_results.keys())
cv_means = [cv_results[n].mean() for n in model_names]
cv_stds = [cv_results[n].std() for n in model_names]

# Plot 1: Accuracy CV
colors = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3']
bars = axes[0].barh(model_names, cv_means, xerr=cv_stds, color=colors,
                     edgecolor='black', capsize=5)
axes[0].set_xlabel('Accuracy (CV)', fontsize=12)
axes[0].set_title('Perbandingan Accuracy Cross-Validation', fontsize=13, fontweight='bold')
# Sesuaikan xlim berdasarkan data
min_val = min(cv_means) - max(cv_stds) - 0.01
axes[0].set_xlim(max(0, min_val), 1.01)

# Tambah label nilai
for i, (mean, std) in enumerate(zip(cv_means, cv_stds)):
    axes[0].text(mean + std + 0.002, i, f'{mean:.4f}', va='center', fontsize=10)

# Plot 2: Train vs Test Accuracy
acc_trains = [eval_results[n]['Accuracy Train'] for n in model_names]
acc_tests = [eval_results[n]['Accuracy Test'] for n in model_names]

x = np.arange(len(model_names))
width = 0.35
axes[1].barh(x - width/2, acc_trains, width, label='Data Latih', color='steelblue', edgecolor='black')
axes[1].barh(x + width/2, acc_tests, width, label='Data Uji', color='coral', edgecolor='black')
axes[1].set_yticks(x)
axes[1].set_yticklabels(model_names)
axes[1].set_xlabel('Accuracy', fontsize=12)
axes[1].set_title('Accuracy: Data Latih vs Data Uji', fontsize=13, fontweight='bold')
axes[1].legend()
min_val2 = min(min(acc_trains), min(acc_tests)) - 0.02
axes[1].set_xlim(max(0, min_val2), 1.01)

plt.tight_layout()
plt.show()

**Temuan Perbandingan:**
- **Accuracy semua model sangat tinggi** (mendekati atau di atas 97%). Ini **bukan berarti semua model hebat**, melainkan karena dataset sangat imbalanced — model yang selalu menebak "Tidak Depresi" (kelas 0) saja sudah mendapat accuracy ~97.4%.
- Yang lebih penting dilihat: **Precision** (dari yang ditebak depresi, berapa yang benar depresi) dan **Recall** (dari semua yang benar depresi, berapa yang berhasil ditemukan).
- Perhatikan selisih **Train vs Test accuracy**:
  - Decision Tree sering punya accuracy latih **sangat tinggi** (bahkan 100%) tapi accuracy uji lebih rendah → ini tanda **overfitting** (menghafal data latih).
  - Random Forest dan XGBoost biasanya punya selisih lebih kecil → lebih **stabil** dan **generalisasi** lebih baik.

### Bonus: Visualisasi Decision Tree (White Box)

Salah satu keunggulan Decision Tree adalah **bisa divisualisasi**. Kita bisa melihat pertanyaan apa yang ditanyakan model untuk membuat keputusan.

In [ ]:
# Visualisasi Decision Tree (kedalaman dibatasi 3 agar terbaca)
pohon = DecisionTreeClassifier(max_depth=3, random_state=42)
pohon.fit(X_train, y_train)

plt.figure(figsize=(20, 10))
plot_tree(pohon,
          feature_names=X_train.columns.tolist(),
          class_names=['Tidak Depresi', 'Depresi'],
          filled=True,
          rounded=True,
          fontsize=10)
plt.title('Decision Tree (max_depth=3)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Feature importance dari Random Forest
rf_model = trained_models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
importances_sorted = importances.sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importances_sorted.plot(kind='barh', color='steelblue', edgecolor='black')
plt.xlabel('Feature Importance', fontsize=12)
plt.title('Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Top 5 fitur terpenting menurut Random Forest:")
for feat, imp in importances_sorted.tail(5).iloc[::-1].items():
    print(f"  {feat}: {imp:.4f}")

## 9. Confusion Matrix Perbandingan

Membandingkan confusion matrix antara **model standar (KNN)** dan **model dengan skor CV terbaik**.

In [ ]:
# Cari model terbaik berdasarkan CV mean
nama_model_terbaik = max(cv_results, key=lambda k: cv_results[k].mean())
model_terbaik = trained_models[nama_model_terbaik]
knn_name = f'KNN (k={k_terbaik})'

print(f"Model standar  : {knn_name}")
print(f"Model terbaik  : {nama_model_terbaik} (CV Mean: {cv_results[nama_model_terbaik].mean():.4f})")

# Prediksi
y_pred_knn_final = trained_models[knn_name].predict(X_test)
y_pred_terbaik = model_terbaik.predict(X_test)

# Confusion Matrix side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_knn = confusion_matrix(y_test, y_pred_knn_final)
ConfusionMatrixDisplay(cm_knn, display_labels=['Tidak Depresi', 'Depresi']).plot(
    cmap='Blues', colorbar=False, ax=axes[0])
axes[0].set_title(f'Model Standar: {knn_name}', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Prediksi')
axes[0].set_ylabel('Aktual')

cm_terbaik = confusion_matrix(y_test, y_pred_terbaik)
ConfusionMatrixDisplay(cm_terbaik, display_labels=['Tidak Depresi', 'Depresi']).plot(
    cmap='Greens', colorbar=False, ax=axes[1])
axes[1].set_title(f'Model Terbaik: {nama_model_terbaik}', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Prediksi')
axes[1].set_ylabel('Aktual')

plt.tight_layout()
plt.show()

# Penjelasan confusion matrix
print("\nPenjelasan Confusion Matrix:")
print("-" * 55)
print("  TN (kiri atas) : Tidak Depresi, ditebak Tidak Depresi (BENAR)")
print("  FP (kanan atas) : Tidak Depresi, ditebak Depresi (SALAH)")
print("  FN (kiri bawah) : Depresi, ditebak Tidak Depresi (SALAH - BERBAHAYA!)")
print("  TP (kanan bawah): Depresi, ditebak Depresi (BENAR)")
print("\nDalam konteks deteksi depresi, FN (False Negative) lebih")
print("berbahaya karena remaja yang depresi tidak terdeteksi.")

## 10. Pemilihan Model & Evaluasi Final

Model terpilih ditentukan berdasarkan **skor cross-validation** (bukan data uji), lalu dievaluasi **sekali** di data uji sebagai laporan akhir. Ini sesuai aturan: data uji hanya dipakai satu kali agar tidak "mengintip soal ujian".

In [ ]:
# Evaluasi final model terpilih
print("=" * 65)
print(f"MODEL TERPILIH: {nama_model_terbaik}")
print("=" * 65)
print(f"Alasan: Memiliki skor cross-validation tertinggi ({cv_results[nama_model_terbaik].mean():.4f})")
print(f"        dengan standar deviasi rendah ({cv_results[nama_model_terbaik].std():.4f})")

# Prediksi final di data uji
y_final_pred = model_terbaik.predict(X_test)

# Metrik final
acc_final = accuracy_score(y_test, y_final_pred)
prec_final = precision_score(y_test, y_final_pred, zero_division=0)
rec_final = recall_score(y_test, y_final_pred, zero_division=0)

print(f"\nHasil Evaluasi di Data Uji (sekali pakai):")
print(f"  Accuracy  : {acc_final:.4f} ({acc_final:.1%})")
print(f"  Precision : {prec_final:.4f} ({prec_final:.1%})")
print(f"  Recall    : {rec_final:.4f} ({rec_final:.1%})")

print(f"\nClassification Report:")
print(classification_report(y_test, y_final_pred, zero_division=0,
                            target_names=['Tidak Depresi', 'Depresi']))

# Confusion Matrix Final
plt.figure(figsize=(6, 5))
cm_final = confusion_matrix(y_test, y_final_pred)
ConfusionMatrixDisplay(cm_final, display_labels=['Tidak Depresi', 'Depresi']).plot(
    cmap='Purples', colorbar=False)
plt.title(f'Confusion Matrix Final: {nama_model_terbaik}', fontsize=13, fontweight='bold')
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.tight_layout()
plt.show()

In [ ]:
# Perbandingan lengkap: Model Standar (KNN) vs Model Terpilih
print("=" * 65)
print("PERBANDINGAN: Model Standar vs Model Terpilih")
print("=" * 65)

knn_acc = accuracy_score(y_test, y_pred_knn_final)
knn_prec = precision_score(y_test, y_pred_knn_final, zero_division=0)
knn_rec = recall_score(y_test, y_pred_knn_final, zero_division=0)

perbandingan_final = pd.DataFrame({
    'Model Standar (KNN)': [cv_results[knn_name].mean(), knn_acc, knn_prec, knn_rec],
    f'Model Terpilih ({nama_model_terbaik})': [cv_results[nama_model_terbaik].mean(), acc_final, prec_final, rec_final],
}, index=['CV Mean', 'Accuracy (Test)', 'Precision (Test)', 'Recall (Test)'])

display(perbandingan_final.style.format('{:.4f}').highlight_max(axis=1, color='lightgreen'))

**Alasan Pemilihan Model:**

Model dipilih berdasarkan **skor cross-validation tertinggi** di data latih, bukan berdasarkan skor data uji. Ini karena:
1. Cross-validation memberikan estimasi yang lebih **robust** (diuji di 5 lipatan berbeda, bukan hanya 1 kali split).
2. Data uji hanya digunakan **sekali** di akhir sebagai laporan, bukan untuk memilih model. Kalau kita memilih model berdasarkan data uji, sama saja dengan "mengintip soal ujian".
3. Skor CV yang tinggi dengan standar deviasi rendah menunjukkan model **konsisten** di berbagai pembagian data.

## 11. Interpretasi & Kesimpulan

### 11.1 Arti Metrik dalam Bahasa Awam

| Metrik | Arti | Analogi |
|---|---|---|
| **Accuracy** | Dari 100 remaja, berapa yang ditebak dengan benar (baik yang depresi maupun tidak). | Dari 100 soal ujian, berapa yang dijawab benar. |
| **Precision** | Dari semua remaja yang **ditebak depresi** oleh model, berapa persen yang **benar-benar depresi**. Precision tinggi = sedikit "alarm palsu". | Dari semua orang yang ditilang polisi, berapa persen yang benar-benar melanggar. |
| **Recall** | Dari semua remaja yang **benar-benar depresi**, berapa persen yang **berhasil ditemukan** oleh model. Recall tinggi = sedikit yang "terlewat". | Dari semua pelanggar lalu lintas, berapa persen yang tertangkap razia. |
| **Confusion Matrix** | Tabel 2×2 yang memecah tebakan model menjadi 4 kotak: benar-benar tepat (TP, TN), alarm palsu (FP), dan yang terlewat (FN). | Tabel nilai ujian yang memecah jawaban jadi: benar-positif, benar-negatif, salah-positif, salah-negatif. |

### 11.2 Konteks Penting

Dalam kasus **deteksi depresi pada remaja**, **Recall lebih penting daripada Precision**, karena:
- **FN (False Negative)** = remaja yang depresi tapi **tidak terdeteksi** → **berbahaya**, karena tidak mendapat bantuan.
- **FP (False Positive)** = remaja yang tidak depresi tapi **ditebak depresi** → **kurang berbahaya**, karena hanya perlu pemeriksaan lanjutan.

Maka, model yang baik untuk kasus ini seharusnya memaksimalkan Recall (mendeteksi sebanyak mungkin remaja depresi), meskipun Precision-nya sedikit menurun.

### 11.3 Keterbatasan Hasil

1. **Dataset sangat imbalanced**: Hanya ~2.6% data berlabel depresi. Semua model cenderung mendapat accuracy tinggi hanya dengan menebak "Tidak Depresi". Skor accuracy yang tinggi **bisa menipu** — ini bukan berarti model benar-benar bagus.
2. **Potensi kebocoran target (target leakage)**: Kolom `stress_level`, `anxiety_level`, dan `addiction_level` secara konseptual sangat dekat dengan `depression_label`. Di dunia nyata, kita mungkin tidak tahu tingkat stres/kecemasan seseorang sebelum mendiagnosis depresi. Ini bisa membuat model terlihat sangat bagus padahal belum tentu berguna di dunia nyata.
3. **Dataset kecil dan kemungkinan sintetis**: 1200 baris dengan pola yang sangat rapi dan tidak ada missing value sama sekali bisa mengindikasikan data yang dibuat (generated), bukan data survei asli. Hasil model mungkin tidak berlaku di populasi nyata.
4. **Tidak ada validasi eksternal**: Model belum diuji pada dataset lain atau populasi yang berbeda.

### 11.4 Kesimpulan

Dari 4 model yang dibandingkan (KNN, Decision Tree, Random Forest, XGBoost), model terbaik dipilih berdasarkan skor cross-validation 5 lipatan di data latih. Semua model menunjukkan accuracy yang tinggi, namun perlu diingat bahwa ini dipengaruhi oleh ketidakseimbangan kelas yang ekstrem.

Untuk penggunaan nyata di masa depan, langkah yang disarankan:
- Mengumpulkan data asli yang lebih banyak, terutama kasus depresi (kelas 1).
- Menggunakan teknik penanganan data imbalanced (misalnya SMOTE, oversampling, atau `class_weight='balanced'`).
- Mempertimbangkan fitur mana yang realistis tersedia sebelum diagnosis.
- Melakukan validasi dengan ahli psikologi atau psikiater.

---
## Pembagian Tugas Kelompok

| Nama | NIM | Kontribusi |
|---|---|---|
| *(isi nama)* | *(isi NIM)* | *(isi kontribusi)* |
| *(isi nama)* | *(isi NIM)* | *(isi kontribusi)* |
| *(isi nama)* | *(isi NIM)* | *(isi kontribusi)* |
| *(isi nama)* | *(isi NIM)* | *(isi kontribusi)* |

**Sumber referensi:**
- Dataset: [Teen Mental Health Dataset — Kaggle](https://www.kaggle.com/datasets/sunil123kumar/social-media-impact-on-mental-health)
- Materi Study Group I-Smile Hari 1–4 (EDA, Regresi Linear, KNN & Scaling, Evaluasi & Perbandingan Model)
- Dokumentasi scikit-learn: https://scikit-learn.org/stable/